In [24]:
import pandas as pd

In [25]:
!pip install chardet


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import chardet

with open("balanced-reviews.txt", "rb") as f:
    result = chardet.detect(f.read())

result


{'encoding': 'UTF-16',
 'confidence': 1.0,
 'language': 'ar',
 'mime_type': 'text/plain'}

In [27]:
df = pd.read_csv("balanced-reviews.txt", delimiter="\t", encoding=result["encoding"]) 

In [28]:
df.to_csv("balanced-reviews.csv", index=False)

In [29]:
df

,no,Hotel name,rating,user type,room type,nights,review
0,2,فندق 72,2,مسافر منفرد,غرفة ديلوكس مزدوجة أو توأم,أقمت ليلة واحدة,“ممتاز”. النظافة والطاقم متعاون.
1,3,فندق 72,5,زوج,غرفة ديلوكس مزدوجة أو توأم,أقمت ليلة واحدة,استثنائي. سهولة إنهاء المعاملة في الاستقبال. ل...
2,16,فندق 72,5,زوج,-,أقمت ليلتين,استثنائي. انصح بأختيار الاسويت و بالاخص غرفه ر...
3,20,فندق 72,1,زوج,غرفة قياسية مزدوجة,أقمت ليلة واحدة,“استغرب تقييم الفندق كخمس نجوم”. لا شي. يستحق ...
4,23,فندق 72,4,زوج,غرفة ديلوكس مزدوجة أو توأم,أقمت ليلتين,جيد. المكان جميل وهاديء. كل شي جيد ونظيف بس كا...
...,...,...,...,...,...,...,...
105693,412050,زوار انترناشيونال,1,أسرة,غرفة قياسية توأم,أقمت ليلة واحدة,“فند”. لا شئ عجبني. طقم العمل سيئ جدالا يوجد ب...
105694,412051,زوار انترناشيونال,2,زوج,غرفة ديلوكس توأم,أقمت ليلتين,“سيئ”. قربه من المسجد النبوي الشريف. استخدام م...
105695,412054,زوار انترناشيونال,2,زوج,غرفة قياسية ثلاثية,أقمت ليلة واحدة,“اسوأ إقامة في الرحلة !”. القرب من الحرم. كل ش...
105696,412058,زوار انترناشيونال,2,أسرة,غرفة قياسية رباعية,أقمت 3 ليالي,“دون المستوى”. قربه من الحرم. كل شيء


In [30]:
import numpy as np
import re
import unicodedata
from collections import Counter

In [31]:
INPUT_PATH  = df
OUTPUT_PATH = "arareviews_clean.csv"

In [32]:
print(f"{len(df):,} rows")
print(f"Columns: {df.columns.tolist()}")
print(f"Rating distribution:\n{df['rating'].value_counts().sort_index()}\n")    

105,698 rows
Columns: ['no', 'Hotel name', 'rating', 'user type', 'room type', 'nights', 'review']
Rating distribution:
rating
1    14382
2    38467
4    26450
5    26399
Name: count, dtype: int64



In [33]:
def map_labels(df: pd.DataFrame) -> pd.DataFrame:
    """
    1-2 → negative
    3   → neutral (none in this dataset but kept for completeness)
    4-5 → positive
    """
    def rating_to_label(r):
        if r <= 2:   return "negative"
        elif r == 3: return "neutral"
        else:        return "positive"

    df["label"] = df["rating"].apply(rating_to_label)
    print("Label distribution (before balancing):")
    print(df["label"].value_counts())
    print()
    return df

In [34]:
df = map_labels(df)

Label distribution (before balancing):
label
negative    52849
positive    52849
Name: count, dtype: int64



In [35]:
def clean_arabic(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # Remove diacritics (tashkeel) — حركات
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)

    # Normalize Alef variants → ا
    text = re.sub(r'[إأآ]', 'ا', text)

    # Normalize Teh Marbuta → ه
    text = re.sub(r'ة', 'ه', text)

    # Normalize Yeh variants → ي
    text = re.sub(r'ى', 'ي', text)

    # Remove English characters
    text = re.sub(r'[a-zA-Z]', '', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove emojis and special unicode
    text = re.sub(r'[^\u0600-\u06FF\u0750-\u077F\s\.\،\!\?]', '', text)

    # Remove punctuation except Arabic punctuation
    text = re.sub(r'[\"\'`]', '', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [36]:
def apply_cleaning(df: pd.DataFrame) -> pd.DataFrame:
    print("Cleaning Arabic text...")
    df["clean_review"] = df["review"].apply(clean_arabic)

    # Remove empty reviews after cleaning
    before = len(df)
    df = df[df["clean_review"].str.len() > 10].reset_index(drop=True)
    after = len(df)
    print(f"Removed {before - after:,} empty/short reviews after cleaning")
    print(f"Remaining: {after:,} reviews\n")
    return df

In [37]:
df = apply_cleaning(df)

Cleaning Arabic text...
Removed 180 empty/short reviews after cleaning
Remaining: 105,518 reviews



In [38]:
print(f"{len(df):,} rows")
print(f"Columns: {df.columns.tolist()}")
print(f"Rating distribution:\n{df['label'].value_counts().sort_index()}\n")  

105,518 rows
Columns: ['no', 'Hotel name', 'rating', 'user type', 'room type', 'nights', 'review', 'label', 'clean_review']
Rating distribution:
label
negative    52785
positive    52733
Name: count, dtype: int64



In [40]:
def balance_dataset(df: pd.DataFrame, samples_per_class: int = 52733) -> pd.DataFrame:
    """
    Undersample to balance classes.
    Use the minority class size as the cap.
    """
    min_count = df["label"].value_counts().min()
    cap = min(samples_per_class, min_count)

    print(f"Balancing dataset to {cap:,} samples per class...")
    balanced = (
        df.groupby("label", group_keys=False)
        .sample(n=cap, random_state=42)
        .reset_index(drop=True)
    )
    print(f"Balanced distribution:")
    print(balanced["label"].value_counts())
    print(f"Total: {len(balanced):,} rows\n")
    return balanced

In [41]:
df = balance_dataset(df)

Balancing dataset to 52,733 samples per class...
Balanced distribution:
label
negative    52733
positive    52733
Name: count, dtype: int64
Total: 105,466 rows



In [42]:
def quality_check(df: pd.DataFrame):
    print("=== Quality Check ===")
    print(f"Total rows: {len(df):,}")
    print(f"Null values: {df['clean_review'].isnull().sum()}")
    print(f"Empty strings: {(df['clean_review'] == '').sum()}")
    print(f"Avg review length: {df['clean_review'].str.len().mean():.1f} chars")
    print(f"Min review length: {df['clean_review'].str.len().min()} chars")
    print(f"Max review length: {df['clean_review'].str.len().max()} chars")
    print()

    print("Sample cleaned reviews per label:")
    for label in df["label"].unique():
        sample = df[df["label"] == label]["clean_review"].iloc[0]
        print(f"  [{label}]: {sample[:100]}")
    print()

In [43]:
quality_check(df)

=== Quality Check ===
Total rows: 105,466
Null values: 0
Empty strings: 0
Avg review length: 133.8 chars
Min review length: 11 chars
Max review length: 3361 chars

Sample cleaned reviews per label:
  [negative]: اسوا فندق في العالم. لايوجد اي شي مميز. الفندق سئ جدا .. الفندق فيه اصلاحات وتكسير ولم ننعم بالراحه 
  [positive]: استثنائي. جديد ونظيف وطاقم العمل ودودين جدا وراقين وتعاملهم جميل ..انصح به وبشده. لاشي



In [44]:
def save_dataset(df: pd.DataFrame, path: str):
    final = df[["clean_review", "label", "rating"]].copy()
    final.columns = ["text", "label", "rating"]
    final.to_csv(path, index=False)
    print(f"Saved cleaned dataset to: {path}")
    print(f"Final shape: {final.shape}")
    return final

In [49]:
save_dataset(df, OUTPUT_PATH)

Saved cleaned dataset to: arareviews_clean.csv
Final shape: (105466, 3)


,text,label,rating
0,اسوا فندق في العالم. لايوجد اي شي مميز. الفندق...,negative,1
1,افشل فندق. . فاشل بكل المقاييس,negative,1
2,ازعاااااج. ولاشي. الازعاج الشديد التي تمنعك من...,negative,2
3,كلام فاضي. لا شي. الله يعين الي بيقعد فيه,negative,2
4,اقامه غير مريحه بسبب ازعاج المراقص. الاطلاله. ...,negative,1
...,...,...,...
105461,مريحه و قيمه. كانت اقامه مريحه جدا و الخدمات ع...,positive,5
105462,ممتازه. لم اتناول وجبه الافطار.,positive,5
105463,جيد. السعر ممتاز جدا. عدم وجود ثلاجه,positive,4
105464,جيد. نظافه الفندق. صغر اللوبي,positive,4
